# LLM Time-Series Anomaly Detection: SFT Pipeline

**参考链接**
- [AnomLLM](https://github.com/Rose-STL-Lab/AnomLLM)
- [Unsloth Vision SFT](https://docs.unsloth.ai/basics/vision-fine-tuning)
- [Unsloth Qwen3-VL](https://docs.unsloth.ai/models/qwen3-vl)
- [vLLM Server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html)

**数据**：AnomLLM 合成数据集，4 个子集（flat-trend / range / point / freq），每个 400 条，长度 1000 单通道时序 + 预渲染 PNG

**流程**：Phase 0-1 环境 → Phase 2-4 Baseline → Phase 5-7 SFT 数据 → Phase 8 训练 → Phase 9 评估 → Phase 10 可选 GRPO

**断点**：BPA / BPB / BPC / BPD 处暂停检查，将观察结果告诉 Claude Code 再继续

---
## 阶段 0. 初始化环境

### Cell 0.1 — 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Cell 0.2 — 定义运行目录

In [ ]:
from pathlib import Path

RUNTIME    = Path("/content/tsad_runtime")
DRIVE_ROOT = Path("/content/drive/MyDrive/tsad_anomaly")

RT_CODE    = RUNTIME / "code"
RT_SFT     = RUNTIME / "sft"
RT_CKPT    = RUNTIME / "checkpoints"
RT_RESULTS = RUNTIME / "results"

DRV_PACK   = DRIVE_ROOT / "packs"
DRV_SFT    = DRIVE_ROOT / "sft"
DRV_CKPT   = DRIVE_ROOT / "checkpoints"
DRV_RESULTS= DRIVE_ROOT / "results"

for p in [RUNTIME, RT_CODE, RT_SFT, RT_CKPT, RT_RESULTS,
          DRV_PACK, DRV_SFT, DRV_CKPT, DRV_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

ANOMLLM = RT_CODE / "AnomLLM"

### Cell 0.3 — 安装依赖

In [ ]:
!pip install -U pip -q
!pip install "unsloth[colab-new]" -q
!pip install vllm openai accelerate bitsandbytes scikit-learn pandas pyyaml datasets matplotlib pillow trl -q

---
## 阶段 1. 获取 AnomLLM 代码和数据

### Cell 1.1 — 克隆仓库

In [ ]:
%%bash
if [ ! -d /content/tsad_runtime/code/AnomLLM ]; then
  git clone https://github.com/Rose-STL-Lab/AnomLLM.git /content/tsad_runtime/code/AnomLLM
fi

In [ ]:
import os
os.environ["PYTHONPATH"] = "/content/tsad_runtime/code/AnomLLM/src"

### Cell 1.2 — 恢复或下载数据

In [ ]:
%%bash
# 优先从 Drive 恢复
if [ -f /content/drive/MyDrive/tsad_anomaly/packs/anomllm_data.tar ]; then
  tar -xf /content/drive/MyDrive/tsad_anomaly/packs/anomllm_data.tar \
      -C /content/tsad_runtime/code/AnomLLM
else
  # 方案 A：从 S3 下载预生成数据
  pip install s5cmd -q
  cd /content/tsad_runtime/code/AnomLLM
  s5cmd --no-sign-request --endpoint-url https://s3-west.nrp-nautilus.io \
      cp "s3://anomllm/data/*" data/

  # 方案 B：本地生成（若 S3 不可用，取消下面注释）
  # cd /content/tsad_runtime/code/AnomLLM && bash synthesize.sh

  # 打包备份到 Drive
  tar -cf /content/drive/MyDrive/tsad_anomaly/packs/anomllm_data.tar \
      -C /content/tsad_runtime/code/AnomLLM data
fi

### Cell 1.3 — 确认数据结构

In [ ]:
import pickle, os

for subset in ["point", "range", "freq", "flat-trend"]:
    pkl = f"/content/tsad_runtime/code/AnomLLM/data/synthetic/{subset}/eval/data.pkl"
    with open(pkl, "rb") as f:
        d = pickle.load(f)
    print(subset, "series:", len(d["series"]), d["series"][0].shape,
          "| has_figs:", os.path.isdir(pkl.replace("data.pkl","figs")))

### Cell 1.4 — 固定数据子集

In [ ]:
DATASETS = ["flat-trend", "range", "point", "freq"]

---
## ⏸ 检查点 A：环境 + 数据就绪

**在此暂停**，运行下面的验证 cell，将结果告诉 Claude Code 后再继续。

**检查什么**
- GPU 型号 / 可用 VRAM / BF16 支持
- 四个子集各 400 条，shape `(1000,1)`，`has_figs: True`
- PNG 目视有折线

**告诉 Claude Code**：GPU 型号、可用 VRAM（GB）、BF16 是否 True、`has_figs` 是否全 True

> VRAM 影响 Cell 8.2 `per_device_train_batch_size`；`has_figs=False` 则需插入渲染 cell

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("Free VRAM:", round(torch.cuda.mem_get_info()[0] / 1e9, 1), "GB")
print("BF16:", torch.cuda.is_bf16_supported())

import unsloth  # 无报错即可
print("unsloth OK")

In [ ]:
from IPython.display import Image
Image("/content/tsad_runtime/code/AnomLLM/data/synthetic/point/eval/figs/001.png")

---
## 阶段 2. VLM Baseline

### Cell 2.1 — 启动本地 Qwen3-VL endpoint（后台运行）

`--served-model-name qwen-local` 必须加：`online_api.py` 把 `--model` 参数直接发给 API，vLLM 只认注册名。

In [ ]:
!python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-VL-8B-Instruct \
  --served-model-name qwen-local \
  --host 127.0.0.1 --port 8000 &

### Cell 2.2 — 写 credentials.yml

In [ ]:
creds = """\
qwen-local:
  api_key: dummy
  base_url: "http://127.0.0.1:8000/v1"
"""
(ANOMLLM / "credentials.yml").write_text(creds)

### Cell 2.3 — 跑 VLM baseline 全量

**内联冒烟**：`point` 子集的 jsonl 出现前几行后，先执行 `head -3 results/synthetic/point/qwen-local/0shot-vision.jsonl`，确认 response 为 `[{"start":...,"end":...},...]` 格式、无报错，再让循环继续。

In [ ]:
%%bash
cd /content/tsad_runtime/code/AnomLLM
for datum in flat-trend range point freq; do
  python src/online_api.py --data "$datum" --model qwen-local --variant 0shot-vision
done

---
## 阶段 3. 统计 Baseline

### Cell 3.1 — Isolation Forest

In [ ]:
%%bash
cd /content/tsad_runtime/code/AnomLLM
for datum in flat-trend range point freq; do
  python src/baselines/isoforest.py --data "$datum" --model isolation-forest
done

---
## 阶段 4. 汇总 Baseline 对比

### Cell 4.1 — result_agg.py 生成指标

In [ ]:
%%bash
cd /content/tsad_runtime/code/AnomLLM
mkdir -p results/agg

python src/result_agg.py --data_name flat-trend --label_name flat-trend-exp \
    --table_caption "Trend anomalies (flat)"
python src/result_agg.py --data_name range --label_name range-exp \
    --table_caption "Out-of-range anomalies"
python src/result_agg.py --data_name point --label_name point-exp \
    --table_caption "Point anomalies"
python src/result_agg.py --data_name freq --label_name freq-exp \
    --table_caption "Frequency anomalies"

### Cell 4.2 — 合并成对比 CSV

In [ ]:
import pickle, pandas as pd

frames = []
for datum in ["flat-trend", "range", "point", "freq"]:
    with open(f"/content/tsad_runtime/code/AnomLLM/results/agg/{datum}.pkl", "rb") as f:
        df = pickle.load(f)
    df.insert(0, "dataset", datum)
    frames.append(df)

baseline_df = pd.concat(frames)
baseline_df.to_csv("/content/tsad_runtime/results/baseline_compare.csv")
print(baseline_df.to_string())

---
## ⏸ 检查点 B：Baseline 完成

**在此暂停**，运行下面的验证 cell，将结果告诉 Claude Code 后再继续。

**检查什么**
- 所有 jsonl 均 400 行（不足则中断，需续跑）
- `qwen-local (0shot-vision)` 各子集 F1 非零（期望 0.2~0.7）
- F1 全 0 说明 response 解析失败，把 CSV 头几行告诉 Claude Code

**告诉 Claude Code**：`baseline_compare.csv` 全文

In [ ]:
%%bash
echo "=== VLM baseline ==="
wc -l /content/tsad_runtime/code/AnomLLM/results/synthetic/*/qwen-local/0shot-vision.jsonl
echo "=== Isolation Forest ==="
wc -l /content/tsad_runtime/code/AnomLLM/results/synthetic/*/isolation-forest/0shot.jsonl

In [ ]:
import pandas as pd
print(pd.read_csv("/content/tsad_runtime/results/baseline_compare.csv").to_string())

---
## 阶段 5. 构建 SFT 样本清单

### Cell 5.1 — 生成 sft_manifest.csv

数据来源：已加载的 eval split，无需重新渲染图像（`figs/` 已有 PNG）。

分层切分：1200 train / 150 val / 150 eval

In [ ]:
import pickle, pandas as pd
from sklearn.model_selection import train_test_split

rows = []
for subset in ["flat-trend", "range", "point", "freq"]:
    pkl_path = f"/content/tsad_runtime/code/AnomLLM/data/synthetic/{subset}/eval/data.pkl"
    with open(pkl_path, "rb") as f:
        d = pickle.load(f)
    anom_type = {"flat-trend": "trend", "range": "range",
                 "point": "point", "freq": "freq"}[subset]
    for i, anom_list in enumerate(d["anom"]):
        intervals = anom_list[0]   # sensor 0
        label = 1 if len(intervals) > 0 else 0
        rows.append({
            "sample_id":   f"{subset}_{i:03d}",
            "subset":      subset,
            "pkl_path":    pkl_path,
            "pkl_idx":     i,
            "image_path":  f"/content/tsad_runtime/code/AnomLLM/data/synthetic/{subset}/eval/figs/{i+1:03d}.png",
            "label":       label,
            "anomaly_type": anom_type if label else "none",
            "intervals":   str(intervals),
        })

manifest = pd.DataFrame(rows)

# stratified split: 1200 train / 150 val / 150 eval
train_df, tmp_df = train_test_split(
    manifest, test_size=0.20, random_state=3407,
    stratify=manifest["subset"].astype(str) + "_" + manifest["label"].astype(str)
)
val_df, eval_df = train_test_split(
    tmp_df, test_size=0.50, random_state=3407,
    stratify=tmp_df["subset"].astype(str) + "_" + tmp_df["label"].astype(str)
)
train_df = train_df.assign(split="train")
val_df   = val_df.assign(split="val")
eval_df  = eval_df.assign(split="eval")

manifest = pd.concat([train_df, val_df, eval_df]).sort_values("sample_id")
manifest.to_csv("/content/tsad_runtime/sft/sft_manifest.csv", index=False)
print(manifest["split"].value_counts())
print(manifest.groupby(["split","label"]).size())

---
## 阶段 6. 教师蒸馏与清洗

### Cell 6.1 — 设置教师模型

In [ ]:
import os
os.environ["DASHSCOPE_API_KEY"] = "sk-xxxx"  # ⚠️ 修改此处：替换为你的百炼 API Key

TEACHER_MODEL = "qwen3.5-plus"
TEACHER_TEMPERATURE = 0.2

### Cell 6.2 — 生成蒸馏数据（只对 train + val 调用教师）

**冒烟模式**：默认只跑 20 条（`.head(20)`）确认质量，确认后去掉 `.head(20)` 跑全量。

In [ ]:
import base64, json, pandas as pd
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

SYSTEM = "You are annotating time-series anomaly plots for supervised fine-tuning. Return strict JSON only."
VISION_USER_TEXT = ("Detect ranges of anomalies in this time series, "
                    "in terms of the x-axis coordinate.\n"
                    "List one by one, in JSON format.\n"
                    "If there are no anomalies, answer with an empty list [].\n\n"
                    "Output template:\n"
                    '[{\"start\": ..., \"end\": ...}, {\"start\": ..., \"end\": ...}...]')
USER = (f"{VISION_USER_TEXT}\n\n"
        "Return a strict JSON object with keys: "
        "is_anomaly (bool), anomaly_type (point|range|freq|trend|none), "
        "intervals ([{\"start\": int, \"end\": int}, ...] or []), rationale (str). "
        "No markdown.")

manifest = pd.read_csv("/content/tsad_runtime/sft/sft_manifest.csv")
sft_df   = manifest[manifest["split"].isin(["train", "val"])]
sft_df   = sft_df.head(20)  # ⚠️ 修改此处：确认质量后删除此行以跑全量

out_path = "/content/tsad_runtime/sft/sft_raw.jsonl"
with open(out_path, "w") as fout:
    for _, row in sft_df.iterrows():
        with open(row["image_path"], "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        resp = client.chat.completions.create(
            model=TEACHER_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": [
                    {"type": "text", "text": USER},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ]},
            ],
            response_format={"type": "json_object"},
            extra_body={"enable_thinking": False},
            temperature=TEACHER_TEMPERATURE,
        )
        result = json.loads(resp.choices[0].message.content)
        result["sample_id"]      = row["sample_id"]
        result["image_path"]     = row["image_path"]
        result["ground_truth"]   = int(row["label"])
        result["split"]          = row["split"]
        fout.write(json.dumps(result, ensure_ascii=False) + "\n")

### Cell 6.3 — 自动清洗（label 一致 + intervals 格式合法才保留）

In [ ]:
import json

def valid_interval(interval):
    return (
        isinstance(interval, dict)
        and isinstance(interval.get("start"), (int, float))
        and isinstance(interval.get("end"), (int, float))
        and interval["start"] < interval["end"]
    )

clean, reject = [], []
with open("/content/tsad_runtime/sft/sft_raw.jsonl") as f:
    for line in f:
        r = json.loads(line)
        intervals = r.get("intervals", [])
        teacher_anom = bool(r.get("is_anomaly", False))
        ground_truth = bool(r["ground_truth"])
        intervals_ok = isinstance(intervals, list) and all(valid_interval(iv) for iv in intervals)
        consistent = intervals_ok and (teacher_anom == bool(intervals))
        if teacher_anom == ground_truth and consistent:
            clean.append(r)
        else:
            reject.append(r)

with open("/content/tsad_runtime/sft/sft_final.jsonl", "w") as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"clean={len(clean)}, reject={len(reject)}, "
      f"keep_rate={len(clean)/(len(clean)+len(reject)):.1%}")

---
## 阶段 7. 转成 Unsloth 训练格式

### Cell 7.1 — 生成 messages 格式 JSONL

In [ ]:
import json

USER_TEXT = VISION_USER_TEXT

writers = {
    "train": open("/content/tsad_runtime/sft/train.jsonl", "w"),
    "val":   open("/content/tsad_runtime/sft/val.jsonl",   "w"),
}

with open("/content/tsad_runtime/sft/sft_final.jsonl") as f:
    for line in f:
        r = json.loads(line)
        assistant = json.dumps(r.get("intervals", []), ensure_ascii=False)
        record = {"messages": [
            {"role": "user", "content": [
                {"type": "text",  "text":  USER_TEXT},
                {"type": "image", "image": r["image_path"]},
            ]},
            {"role": "assistant", "content": assistant},
        ]}
        writers[r["split"]].write(json.dumps(record, ensure_ascii=False) + "\n")

for f in writers.values():
    f.close()

### Cell 7.2 — eval split 单独转（导出评估元数据）

In [ ]:
import json, pandas as pd

manifest = pd.read_csv("/content/tsad_runtime/sft/sft_manifest.csv")
eval_rows = manifest[manifest["split"] == "eval"]

with open("/content/tsad_runtime/sft/eval.jsonl", "w") as f:
    for _, row in eval_rows.iterrows():
        record = {
            "sample_id":   row["sample_id"],
            "image_path":  row["image_path"],
            "label":       int(row["label"]),
            "anomaly_type": row["anomaly_type"],
            "intervals":   row["intervals"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

---
## 阶段 8. SFT 训练（Unsloth 官方范式）

### Cell 8.1 — 加载模型

In [ ]:
from datasets import load_dataset
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules="all-linear",
)

train_dataset = load_dataset("json", data_files=str(RT_SFT / "train.jsonl"), split="train")
val_dataset   = load_dataset("json", data_files=str(RT_SFT / "val.jsonl"),   split="train")

### Cell 8.2 — 训练

**冒烟模式**：默认 `max_steps=5`，确认无 OOM、loss 正常后注释掉该行跑完整训练。

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

sft_args = SFTConfig(
    output_dir=str(RT_CKPT / "qwen3vl-tsad"),
    max_length=None,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # ⚠️ 修改此处：OOM 则改为 1
    gradient_accumulation_steps=8,
    learning_rate=1e-4,             # ⚠️ 修改此处：loss=nan 则降至 2e-5
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    max_steps=5,                    # ⚠️ 修改此处：冒烟通过后删除或注释此行跑完整训练
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
)

trainer.train()

---
## ⏸ 检查点 C：SFT 数据就绪 + 训练冒烟

**在此暂停**（`max_steps=5` 跑完后），运行下面的验证 cell，将结果告诉 Claude Code 后再继续。

**检查什么**
- 蒸馏：`is_anomaly` 与 `ground_truth` 一致率 ≥ 70%，保留率 ≥ 70%
- 格式：`image exists: True`，assistant content 可被 `json.loads` 解析为区间列表
- 冒烟：无 OOM，第 1 步 loss 在 1.5~4（不是 nan 或 0）

**告诉 Claude Code**：蒸馏保留率、冒烟前 5 步 loss、是否有报错

> OOM → `per_device_train_batch_size=1`；loss=nan → `learning_rate` 降至 `2e-5`；一致率 < 50% → 修改 prompt 重跑蒸馏
>
> 冒烟通过后去掉 Cell 8.2 中的 `max_steps=5`，重新运行 Cell 8.2 跑完整训练，再跑 Cell 8.3 导出

In [ ]:
import json, os

# 蒸馏质量：前 5 条
print("=== 蒸馏样本（前 5 条）===")
with open("/content/tsad_runtime/sft/sft_raw.jsonl") as f:
    for _ in range(5):
        print(json.dumps(json.loads(f.readline()), indent=2, ensure_ascii=False))

# 格式验证
print("\n=== 格式验证 ===")
with open("/content/tsad_runtime/sft/train.jsonl") as f:
    r = json.loads(f.readline())
user_content = r["messages"][0]["content"]
image_item = next(item for item in user_content if item["type"] == "image")
assistant = json.loads(r["messages"][1]["content"])
assistant_ok = isinstance(assistant, list) and all(
    isinstance(iv, dict) and {"start", "end"} <= set(iv)
    for iv in assistant
)
print("image exists:", os.path.exists(image_item["image"]))
print("assistant json ok:", assistant_ok)
print("assistant preview:", assistant[:2])

### Cell 8.3 — 导出模型

**完整训练结束后**再运行此 cell。

In [ ]:
model.save_pretrained_merged(RT_SFT / "qwen3vl-tsad-merged", tokenizer)
model.save_pretrained(RT_SFT / "qwen3vl-tsad-adapter")

---
## 阶段 9. 对比 SFT 与 Baseline

baseline 在 400 条上算过指标，SFT eval 只有 150 条（跨 4 个子集）。**必须把 baseline 结果过滤到相同的 150 条**，才能公平对比。

### Cell 9.1 — 启动 SFT 模型 vLLM server 并推理

In [ ]:
!python -m vllm.entrypoints.openai.api_server \
  --model /content/tsad_runtime/sft/qwen3vl-tsad-merged \
  --served-model-name sft-model \
  --host 127.0.0.1 --port 8001 &

In [ ]:
import yaml

creds_path = ANOMLLM / "credentials.yml"
creds = yaml.safe_load(creds_path.read_text()) or {}
creds["sft-model"] = {"api_key": "dummy", "base_url": "http://127.0.0.1:8001/v1"}
creds_path.write_text(yaml.safe_dump(creds, sort_keys=False))

In [ ]:
%%bash
cd /content/tsad_runtime/code/AnomLLM
for datum in flat-trend range point freq; do
  python src/online_api.py --data "$datum" --model sft-model --variant 0shot-vision
done

### Cell 9.2 — 在相同的 eval 150 条上计算 SFT 与 baseline 指标

In [ ]:
import json, sys, pickle, numpy as np, pandas as pd
sys.path.insert(0, "/content/tsad_runtime/code/AnomLLM/src")
from utils import compute_metrics, interval_to_vector, load_results

# 1. 读取 eval split，建立 {subset: [pkl_idx, ...]} 映射
eval_manifest = pd.read_csv("/content/tsad_runtime/sft/sft_manifest.csv")
eval_manifest = eval_manifest[eval_manifest["split"] == "eval"]
eval_indices = {}
for subset, grp in eval_manifest.groupby("subset"):
    eval_indices[subset] = grp["pkl_idx"].tolist()

# 2. 加载 eval 数据集的 ground truth
gt_map = {}
for subset in ["flat-trend", "range", "point", "freq"]:
    with open(f"/content/tsad_runtime/code/AnomLLM/data/synthetic/{subset}/eval/data.pkl","rb") as f:
        d = pickle.load(f)
    for idx in eval_indices.get(subset, []):
        intervals = d["anom"][idx][0]
        gt_map[(subset, idx)] = interval_to_vector(
            [{"start": s, "end": e} for s, e in intervals]
        ).flatten()

# 3. 对每个方法提取 eval 样本的预测，计算指标
def eval_metrics(method_label, result_fn, eval_indices_map, gt_map):
    results_raw = load_results(result_fn, raw=False)
    rows = []
    for subset, idxs in eval_indices_map.items():
        for idx in idxs:
            pred = results_raw[idx]
            gt   = gt_map[(subset, idx)]
            if pred is None:
                pred = np.zeros_like(gt)
            m = compute_metrics(gt.reshape(-1,1), pred.reshape(-1,1).astype(int))
            rows.append({"method": method_label, "subset": subset, **m})
    return pd.DataFrame(rows)

BASE_DIR = "/content/tsad_runtime/code/AnomLLM/results/synthetic"
all_frames = []
for subset in ["flat-trend", "range", "point", "freq"]:
    for method, fn in [
        ("isolation-forest", f"{BASE_DIR}/{subset}/isolation-forest/0shot.jsonl"),
        ("qwen-local-0shot",  f"{BASE_DIR}/{subset}/qwen-local/0shot-vision.jsonl"),
        ("sft-0shot",         f"{BASE_DIR}/{subset}/sft-model/0shot-vision.jsonl"),
    ]:
        all_frames.append(eval_metrics(method, fn, {subset: eval_indices[subset]}, gt_map))

result_df = pd.concat(all_frames)
summary = result_df.groupby("method")[["f1", "affi f1"]].mean().round(3)
print(summary)
summary.to_csv("/content/tsad_runtime/results/sft_eval_metrics.csv")

---
## ⏸ 检查点 D：最终对比

**在此暂停**，查看上方 summary 表。

**检查什么**
- SFT 是否比 VLM zero-shot 有提升（F1 高 ≥ 0.03 为可见提升）
- 有无某个子集明显退步

**告诉 Claude Code**：对比表全文，以及是否考虑 GRPO

> 进入 GRPO 需满足：SFT F1 高 ≥ 0.05、蒸馏保留率 ≥ 70%、还有剩余 session 时间。
> 告知奖励函数偏好（label 正确 / region 精确 / 两者结合），Claude Code 届时生成 GRPO cell。

---
## 阶段 10. 可选 GRPO

条件：baseline 对比完成 + SFT 有可见提升 + 蒸馏数据质量稳定。

沿用 Unsloth 官方 VLM RL/GRPO 范式，同一份 `train/val/eval` 数据。

**到达此阶段时，将检查点 D 的结果告诉 Claude Code，由其生成 GRPO 训练 cell。**